# Import Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import randint

# Sklearn preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Sklearn model
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV

# Classification Model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# Metrices
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Warnings
import warnings
warnings.filterwarnings("ignore")


# 1. Data Loading

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/AMZobaer/datasets/main/HR-Employee-Attrition.csv')
df.head(10)

In [ ]:
print("Shape:", df.shape)

In [ ]:
print("\nColumns:", df.columns.tolist())
print("\nTarget distribution (Attrition):")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# ydata profiling
from ydata_profiling import ProfileReport
profile = ProfileReport( df , title="Employee Attrition prediction", explorative = True  )

profile.to_file("ydata.html")

# 2. Data Preprocessing

In [ ]:
# Check null values
total_nulls = df.isnull().sum().sum()
print("Total Null Values in Dataset:", total_nulls)

In [ ]:
# Check duplicated values
total_duplicates = df.duplicated().sum()
print("Total Duplicates in Dataset:", total_duplicates)

In [ ]:
# Drop useless columns
df.drop([
    'EmployeeCount',
    'EmployeeNumber',
    'Over18',
    'StandardHours'
], axis=1, inplace=True)

# New shape
print("New Shape: ", df.shape)

In [ ]:
# Encode target
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

In [ ]:
# Check class imbalance
df['Attrition'].value_counts()

In [ ]:
# Feature selection
X = df.drop('Attrition', axis=1)
y = df['Attrition']

In [ ]:
# Categorical and Numerical Columns
num_cols = X.select_dtypes(exclude='object').columns
cat_cols = X.select_dtypes(include='object').columns

print("Categorical:", cat_cols.to_list())
print("Numerical:", num_cols.to_list())

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 3. Pipeline Creation

In [ ]:
# For numerical features
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [ ]:
# For categorical features
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [ ]:
# Preprocessor (Combine both Numerical and categorical features)
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# 4. Primary Model Selection

Instead of selecting a model based only on theoretical assumptions, multiple classification algorithms (Logistic Regression, Random Forest, Gradient Boosting, XGBoost, and AdaBoost) will be trained and evaluated using stratified cross-validation. The model achieving the highest F1-score will then be selected for further hyperparameter tuning and final evaluation.

In [ ]:
# Baselines
logreg = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=42)

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42)

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight= (y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1)

adab = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42)

In [ ]:
models = {
    'Logistic Regression': logreg,
    'Random Forest': rf,
    'Gradient Boosting': gb,
    'XGBoost': xgb,
    'AdaBoost': adab}

In [ ]:
pipeline = {
    name: Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    for name, model in models.items()
}

# 5. Model Training

In [ ]:
for name, pipe in pipeline.items():
    pipe.fit(X_train, y_train)

print("Trained: ", list(pipeline.keys()))

# 06. Cross-Validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

In [ ]:
results = {}
for name, pipe in pipeline.items():
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1)
    results[name] = scores

In [ ]:
rows = []
for name, scores in results.items():
    row = {
       "Model": name,
        "Accuracy Mean": f"{np.mean(scores['test_accuracy']):.4f}",
        "Accuracy Std": f"{np.std(scores['test_accuracy']):.4f}",
        "Precision Mean": f"{np.mean(scores['test_precision']):.4f}",
        "Precision Std": f"{np.std(scores['test_precision']):.4f}",
        "Recall Mean": f"{np.mean(scores['test_recall']):.4f}",
        "Recall Std": f"{np.std(scores['test_recall']):.4f}",
        "F1 Mean": f"{np.mean(scores['test_f1']):.4f}",
        "F1 Std": f"{np.std(scores['test_f1']):.4f}",
        "ROC AUC Mean": f"{np.mean(scores['test_roc_auc']):.4f}",
        "ROC AUC Std": f"{np.std(scores['test_roc_auc']):.4f}",
    }
    rows.append(row)

cv_table = pd.DataFrame(rows).sort_values(by="F1 Mean", ascending=False).reset_index(drop=True)
cv_table

Based on 5-fold Stratified Cross-Validation, XGBoost achieved the highest average F1-score and was selected as the primary model for further tuning.

# 07. Hyperparameter Tuning

In [ ]:
# Hyperparameters
param_dist = {
    "model__n_estimators": randint(100, 501),
    "model__max_depth": [2, 3, 4, 5],
    "model__learning_rate": [0.03, 0.05],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

In [ ]:
# Static parameters
xgb_base = XGBClassifier(
    scale_pos_weight= (y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [ ]:
# Pipeline for XGBoost including preprocessing
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_base)
])

In [ ]:
# Randomized Search for Best Hyperparameters
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_iter=30,
    n_jobs=-1,
    verbose=1,
)

random_search_xgb.fit(X_train, y_train)

# 08. Best Model Selection

In [ ]:
# Best parameters
print("Best Parameters:")
print(random_search_xgb.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(random_search_xgb.best_score_)

In [ ]:
# Best Model
best_model = random_search_xgb.best_estimator_

model_name = best_model.named_steps["model"].__class__.__name__

print("Best Model:", model_name)

# 09. Model Performance Evaluation

In [ ]:
# Evalute thr test set
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

In [ ]:
# Metrices
print("Test Results")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1       :", f1_score(y_test, y_pred))
print("ROC AUC  :", roc_auc_score(y_test, y_proba))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(6,5))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            cbar=False,
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'])

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix of XGBoost Model on Test Set')
plt.tight_layout()
plt.show()


The tuned XGBoost model was selected as the final model as it achieved the highest cross-validation F1-score (0.55) among all evaluated models.

On the unseen test set, the model achieved an accuracy of approximately 83%, with a precision of 0.468, recall of 0.468, and an F1-score of 0.468. While the F1-score decreased compared to cross-validation performance, the model demonstrates a balanced trade-off between precision and recall under class imbalance.

The confusion matrix indicates that the model correctly classified 222 non-attrition cases and 22 attrition cases, with an equal number of false positives and false negatives (25 each), suggesting stable predictive behavior.

Additionally, the ROC-AUC score of 0.783 reflects good discriminative ability between employees who leave and those who stay.

Overall, the XGBoost model provides reliable and balanced performance for predicting employee attrition in this dataset, making it a suitable choice for deployment.